# 4. Run an experiment for the installed method

Experiments can run for hours, so the runner is started in the background on `control` (it keeps running if
this notebook or the Jupyter session is closed). Use the status cell to follow progress.

Every run is stored on `control` in `~/experiment-results/<method>-<timestamp>/`, together with its log
(`runner.log`) and a snapshot of its configuration (`run-configuration.json`: settings, branch and commit, images,
node sites/hosts and pod placement). This folder is kept when DYNAMOS is reset or reinstalled, so results stay
available over SSH. The download cell additionally copies a run into `results/` here as a zip file.

In [ ]:
METHOD = "baseline"   # must match the method installed by 03_install_method
CYCLES = None         # None = runner default (120 local steps; 12 rounds for fed-encrypt-dynamic)
RUNS = 1
EXTRA_ARGS = ""       # e.g. "--communication-frequency 10 --sample-batch-size 64" or "--policy-removal 4 --policy-reintroduction 8"

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

import deployment_config as cfg
import fabric_helpers as fh
from fabric_helpers import run, run_script

fablib = fablib_manager()
slice_ = fh.get_slice(fablib)
nodes = fh.nodes_by_name(slice_)
control = nodes[cfg.CONTROL_NODE]
dynamos = nodes[cfg.DYNAMOS_NODE]
print(f"Slice {slice_.get_name()}: {slice_.get_state()}, lease end {slice_.get_lease_end()}")

In [ ]:
import datetime
import shlex

assert METHOD in cfg.METHODS, f"Unknown method {METHOD!r}"
installed = run(control, f"cat {cfg.INSTALLED_METHOD_FILE} 2>/dev/null || true", quiet=True).strip()
assert installed == METHOD, f"Installed method is {installed or 'none'!r}, not {METHOD!r}; run 03_install_method first"

checkout = cfg.checkout_dir(METHOD)
python = f"$HOME/{cfg.EXPERIMENT_VENV}/bin/python"
run(control, f"{python} -m pip install -q -r $HOME/{checkout}/{cfg.METHODS[METHOD]['requirements']}", quiet=True)
print(run(control, f"{python} $HOME/{checkout}/{cfg.METHODS[METHOD]['runner']} --help", quiet=True))

## Start the experiment in the background

In [ ]:
running = run(control, f"test -f {cfg.CURRENT_RUN_FILE} && kill -0 $(cut -d' ' -f2 {cfg.CURRENT_RUN_FILE}) 2>/dev/null && echo yes || echo no", quiet=True).strip()
assert running == "no", "An experiment is still running; check or stop it below first"

run_id = f"{METHOD}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"
output_dir = f"$HOME/{cfg.RESULTS_DIR}/{run_id}"
log_file = f"{output_dir}/runner.log"
args = f"--runs {RUNS} --output-dir {output_dir}"
if CYCLES:
    args += f" --cycles {CYCLES}"
if EXTRA_ARGS:
    args += f" {EXTRA_ARGS}"

settings = {"run_id": run_id, "method": METHOD, "cycles": CYCLES, "runs": RUNS, "extra_args": EXTRA_ARGS,
            "command": f"{cfg.METHODS[METHOD]['runner']} {args}"}
fh.save_run_configuration(control, output_dir, fh.run_configuration(slice_, control, checkout, settings))

start = (
    f"cd $HOME/{checkout} || exit 1; "
    f"nohup setsid {python} {cfg.METHODS[METHOD]['runner']} {args} > {log_file} 2>&1 < /dev/null & "
    f"echo \"{run_id} $!\" > $HOME/{cfg.CURRENT_RUN_FILE}; sleep 5; kill -0 $! || {{ tail -n 30 {log_file}; exit 1; }}"
)
run(control, start, quiet=True)
print(f"Started {run_id}: {cfg.METHODS[METHOD]['runner']} {args}")

## Status (rerun whenever you like)

In [ ]:
print(run(control, f"""
read run_id pid < {cfg.CURRENT_RUN_FILE}
if kill -0 "$pid" 2>/dev/null; then echo "$run_id: RUNNING (pid $pid)"; else echo "$run_id: FINISHED"; fi
log=$HOME/{cfg.RESULTS_DIR}/$run_id/runner.log; [ -f "$log" ] || log=$HOME/{cfg.LOGS_DIR}/$run_id.log
echo; tail -n 30 "$log"
""", quiet=True))

### Intermediate results (optional)

Reads the latest `result.json` the runner saved for the current run (refreshed every 5 s) and lists the sync rounds
completed so far. With the updated `semi-overlap-fed-bcd` API gateway, each round shows its accuracy one round after it
finishes (the server reports it at the next aggregation); the last round's accuracy, and on the other methods all
accuracies, only appear when the run is done.

In [ ]:
import json
import shlex

RUN_ID = None   # None = the current (or last started) run
SHOW_ROUNDS = 10

run_id_ = RUN_ID or run(control, f"cut -d' ' -f1 {cfg.CURRENT_RUN_FILE}", quiet=True).strip()
result_file = run(control, f"ls -t $HOME/{cfg.RESULTS_DIR}/{run_id_}/*/run-*/result.json 2>/dev/null | head -n 1",
                  check=False, quiet=True).strip()
assert result_file, f"No result.json yet for {run_id_}; the request may still be starting"
try:
    result = json.loads(run(control, f"cat {shlex.quote(result_file)}", quiet=True))
except json.JSONDecodeError:
    raise RuntimeError("result.json was being rewritten; rerun this cell") from None

rounds = result.get("results") or []
with_accuracy = [r for r in rounds if r.get("accuracy") is not None]
print(f"{run_id_}: status {result.get('status')}, {len(rounds)} sync rounds completed, "
      f"{len(with_accuracy)} with accuracy\n({result_file})\n")
for r in rounds[-SHOW_ROUNDS:]:
    accuracy = r.get("accuracy")
    latency = (r.get("timing") or {}).get("cycle_latency_ms")
    print(f"  round {r.get('train_round'):>5}   accuracy {'-' if accuracy is None else f'{accuracy:6.2f} %':>8}   "
          f"round time {'-' if latency is None else f'{latency / 1000:.2f} s'}")

## Stop the running experiment (only if needed)

In [ ]:
STOP = False

if STOP:
    print(run(control, f"read run_id pid < {cfg.CURRENT_RUN_FILE}; kill -TERM -- -$pid && echo stopped $run_id", check=False, quiet=True))

## Results stored on `control`

Lists every run kept on `control` and prints the SSH command to reach them.

In [ ]:
print(run(control, f"cd $HOME/{cfg.RESULTS_DIR} 2>/dev/null && du -sh -- */ || echo 'No results yet'", check=False, quiet=True))
print(f"\nOn control: ~/{cfg.RESULTS_DIR}/<run_id>/")
print(f"SSH: {control.get_ssh_command()}")

## Download a run into `results/` (zip, opens in Windows)

In [ ]:
import os
import zipfile

RUN_ID = None   # None = the most recent run; or a folder name listed above, e.g. "baseline-20261110-141453"

run_id = RUN_ID or run(control, f"cut -d' ' -f1 {cfg.CURRENT_RUN_FILE}", quiet=True).strip()
remote_dir = f"{cfg.RESULTS_DIR}/{run_id}"
legacy_log = f"{cfg.LOGS_DIR}/{run_id}.log"
archive = f"{cfg.REMOTE_TOOLS_DIR}/{run_id}.zip"
# Runs started before the log moved into the run folder keep it in ~/experiment-logs; copy it in.
run(control, f"cd $HOME && test -d {remote_dir} "
             f"&& {{ [ -f {remote_dir}/runner.log ] || [ ! -f {legacy_log} ] || cp {legacy_log} {remote_dir}/runner.log; }} "
             f"&& rm -f {archive} && cd {cfg.RESULTS_DIR} && python3 -m zipfile -c $HOME/{archive} {run_id}", quiet=True)

os.makedirs("results", exist_ok=True)
local_zip = os.path.join("results", f"{run_id}.zip")
control.download_file(local_file_path=local_zip, remote_file_path=archive)
with zipfile.ZipFile(local_zip) as archive_file:
    archive_file.extractall("results")
print(f"Saved to results/{run_id}/ and results/{run_id}.zip")